## Model Selection: LLama 3.2 1B

### Preparation

In [1]:
!pip install -q -U transformers peft trl bitsandbytes datasets scikit-learn accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.7 MB/s eta 0:00:00


### Execution

In [3]:
import os
import gc
import time
import torch
import random
import numpy as np

from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import f1_score, accuracy_score, classification_report
from huggingface_hub import login

# =========================================================
# 1. GPU CHECK + CLEANUP
# =========================================================
if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. Enable T4 GPU in Colab.")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("GPU ready.")

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

# =========================================================
# 2. COLAB SETUP (optional)
# =========================================================
try:
    from google.colab import drive, userdata

    drive.mount("/content/drive", force_remount=True)

    REPO_DIR = "/content/drive/MyDrive/progettoLLM/CLARITY"
    HF_CACHE = "/content/drive/MyDrive/progettoLLM/hf_cache"

    os.makedirs(HF_CACHE, exist_ok=True)
    os.environ["HF_HOME"] = HF_CACHE
    os.chdir(REPO_DIR)

    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)

    print("Drive + HuggingFace configured.")

except Exception as e:
    print("Colab setup skipped:", e)

# =========================================================
# 3. DATASET
# =========================================================
VALID_LABELS = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
label2id = {label: i for i, label in enumerate(VALID_LABELS)}
MAJORITY_CLASS_ID = label2id["Ambivalent"]

model_id = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def format_prompt(example, is_inference=False):
    prompt = (
        "You are an expert political analyst. "
        "Classify the clarity of the following interview answer.\n"
        "Choose EXACTLY ONE label:\n"
        "[Clear Reply, Ambivalent, Clear Non-Reply]\n\n"
        f"Question: {example['question']}\n"
        f"Answer: {example['interview_answer']}\n\n"
        "Reasoning: Let's think step by step.\n"
        "Label: "
    )

    if not is_inference:
        prompt += example["clarity_label"] + tokenizer.eos_token

    return prompt


print("Loading dataset...")

train_dataset = load_from_disk("dataset/QEvasion/train")
test_dataset  = load_from_disk("dataset/QEvasion/test")

train_dataset = train_dataset.map(lambda x: {"text": format_prompt(x, False)})
test_dataset  = test_dataset.map(lambda x: {"text": format_prompt(x, False)})

# =========================================================
# 4. LOAD MODEL
# =========================================================
print("Loading 4-bit model...")

# Use bfloat16 everywhere — Llama 3.2's native dtype.
# This avoids the FP16 grad scaler / BF16 tensor mismatch on T4.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=torch.bfloat16,
)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

# =========================================================
# 5. LORA
# =========================================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

# =========================================================
# 6. TRAINING CONFIG
# =========================================================
training_args = SFTConfig(
    output_dir="./llama_clarity_results",

    dataset_text_field="text",
    max_length=512,

    num_train_epochs=1,
    learning_rate=2e-4,
    warmup_steps=50,
    weight_decay=0.01,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,

    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="no",

    fp16=False,
    bf16=True,          # matches Llama 3.2's native dtype

    gradient_checkpointing=True,
    dataloader_num_workers=2,

    optim="paged_adamw_32bit",
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=train_dataset,
    args=training_args,
)

# =========================================================
# 7. TRAIN
# =========================================================
print("Starting training...")

start_time = time.time()

trainer.train()

end_time = time.time()
training_minutes = (end_time - start_time) / 60

print(f"Training finished in {training_minutes:.2f} min")

# =========================================================
# 8. INFERENCE
# =========================================================
def extract_label(text):
    lower = text.lower()
    found = []

    for label in VALID_LABELS:
        if label.lower() in lower:
            found.append(label)

    if found:
        return label2id[found[-1]]

    return MAJORITY_CLASS_ID


print("Evaluating...")

peft_model.eval()

true_labels = []
pred_labels = []

for item in test_dataset:
    prompt = format_prompt(item, True)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to("cuda")

    with torch.no_grad():
        outputs = peft_model.generate(
            **inputs,
            max_new_tokens=75,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]

    generated = tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True,
    )

    pred_id = extract_label(generated)
    true_id = label2id[item["clarity_label"]]

    pred_labels.append(pred_id)
    true_labels.append(true_id)

# =========================================================
# 9. METRICS
# =========================================================
accuracy    = accuracy_score(true_labels, pred_labels)
macro_f1    = f1_score(true_labels, pred_labels, average="macro")
weighted_f1 = f1_score(true_labels, pred_labels, average="weighted")

peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3)

print("\n===================================")
print("FINAL RESULTS")
print("===================================")
print(f"Accuracy:      {accuracy:.4f}")
print(f"Macro F1:      {macro_f1:.4f}")
print(f"Weighted F1:   {weighted_f1:.4f}")
print("-----------------------------------")
print(f"Training Time: {training_minutes:.2f} min")
print(f"Peak VRAM:     {peak_vram:.2f} GB")
print("===================================\n")

print(
    classification_report(
        true_labels,
        pred_labels,
        target_names=VALID_LABELS,
    )
)

GPU ready.
Mounted at /content/drive
Drive + HuggingFace configured.
Loading dataset...
Loading 4-bit model...


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


...

Step,Training Loss
10,3.345545
20,2.953269
30,2.561177


Step,Training Loss
10,3.345545
20,2.953269
30,2.561177
40,2.412599
50,2.331703
60,2.364622
70,2.307598
80,2.276227
90,2.268745
100,2.297168


Training finished in 97.19 min
Evaluating...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



FINAL RESULTS
Accuracy:      0.6818
Macro F1:      0.4054
Weighted F1:   0.6073
-----------------------------------
Training Time: 97.19 min
Peak VRAM:     4.12 GB

                 precision    recall  f1-score   support

    Clear Reply       0.50      0.13      0.20        79
     Ambivalent       0.70      0.96      0.81       206
Clear Non-Reply       0.50      0.13      0.21        23

       accuracy                           0.68       308
      macro avg       0.57      0.40      0.41       308
   weighted avg       0.63      0.68      0.61       308

